In [3]:
import pandas as pd
import mysql.connector

# CSV path
csv_file = r"C:\Users\DELL\Downloads\project1_df.csv"

# Load CSV in chunks
chunk_size = 10000

# Connect to MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="password="YOUR_PASSWORD",
    database="ecommerce"
)
cursor = conn.cursor()

for chunk in pd.read_csv(csv_file, chunksize=chunk_size):
    # Convert data types
    chunk['CID'] = chunk['CID'].astype(str)
    chunk['TID'] = chunk['TID'].astype(str)
    # Correct date parsing
    chunk['Purchase Date'] = pd.to_datetime(chunk['Purchase Date'], dayfirst=True).dt.date
    chunk['Discount Amount (INR)'] = chunk['Discount Amount (INR)'].astype(float)
    chunk['Gross Amount'] = chunk['Gross Amount'].astype(float)
    chunk['Net Amount'] = chunk['Net Amount'].astype(float)

    # Replace NaN in Discount Name with None
    chunk['Discount Name'] = chunk['Discount Name'].where(pd.notnull(chunk['Discount Name']), None)

    # Prepare batch insert
    data = [
        (
            row['CID'], row['TID'], row['Gender'], row['Age Group'], row['Purchase Date'],
            row['Product Category'], row['Discount Availed'], row['Discount Name'],
            row['Discount Amount (INR)'], row['Gross Amount'], row['Net Amount'],
            row['Purchase Method'], row['Location']
        )
        for _, row in chunk.iterrows()
    ]

    # Batch insert
    cursor.executemany("""
        INSERT INTO ecommerce_orders (
            cid, tid, gender, age_group, purchase_date, product_category, discount_availed,
            discount_name, discount_amount_inr, gross_amount, net_amount,
            purchase_method, location
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, data)
    conn.commit()
    print(f"Inserted {len(chunk)} rows...")

Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 5000 rows...
